# 파이토치로 구현하는 신경망

## 신경망 모델 구현하기

In [ ]:
import torch # 라이브러리 임포트
import numpy # 라이브러리 임포트
from sklearn.datasets import make_blobs # 라이브러리 임포트
import matplotlib.pyplot as plt # 라이브러리 임포트

In [ ]:
n_dim = 2 # 변수 설정
x_train, y_train = make_blobs(n_samples=80, n_features=n_dim, centers=[[1,1],[-1,-1],[1,-1],[-1,1]], shuffle=True, cluster_std=0.3) # 데이터셋 로드
x_test, y_test = make_blobs(n_samples=20, n_features=n_dim, centers=[[1,1],[-1,-1],[1,-1],[-1,1]], shuffle=True, cluster_std=0.3) # 데이터셋 로드

In [ ]:
def label_map(y_, from_, to_): # 함수 정의
    y = numpy.copy(y_) # 변수 설정
    for f in from_: # 반복문 수행
        y[y_ == f] = to_ # 변수 설정
    return y # 결과 반환

y_train = label_map(y_train, [0, 1], 0) # 변수 설정
y_train = label_map(y_train, [2, 3], 1) # 변수 설정
y_test = label_map(y_test, [0, 1], 0) # 변수 설정
y_test = label_map(y_test, [2, 3], 1) # 변수 설정

In [ ]:
def vis_data(x,y = None, c = 'r'): # 함수 정의
    if y is None: # 조건문 수행
        y = [None] * len(x) # 변수 설정
    for x_, y_ in zip(x,y): # 반복문 수행
        if y_ is None: # 조건문 수행
            plt.plot(x_[0], x_[1], '*',markerfacecolor='none', markeredgecolor=c) # 데이터 시각화
        else: # 코드 실행
            plt.plot(x_[0], x_[1], c+'o' if y_ == 0 else c+'+') # 데이터 시각화

plt.figure() # 그래프 객체 생성
vis_data(x_train, y_train, c='r') # 변수 설정
plt.show() # 그래프 출력

In [ ]:
x_train = torch.FloatTensor(x_train) # 텐서 변환
print(x_train.shape) # 결과 출력
x_test = torch.FloatTensor(x_test) # 텐서 변환
y_train = torch.FloatTensor(y_train) # 텐서 변환
y_test = torch.FloatTensor(y_test) # 텐서 변환

In [ ]:
class NeuralNet(torch.nn.Module): # 클래스 정의
        def __init__(self, input_size, hidden_size): # 함수 정의
            super(NeuralNet, self).__init__() # 코드 실행
            self.input_size = input_size # 변수 설정
            self.hidden_size  = hidden_size # 변수 설정
            self.linear_1 = torch.nn.Linear(self.input_size, self.hidden_size) # 선형층 정의
            self.relu = torch.nn.ReLU() # ReLU 활성화 함수
            self.linear_2 = torch.nn.Linear(self.hidden_size, 1) # 선형층 정의
            self.sigmoid = torch.nn.Sigmoid() # Sigmoid 활성화 함수
            
        def forward(self, input_tensor): # 함수 정의
            linear1 = self.linear_1(input_tensor) # 변수 설정
            relu = self.relu(linear1) # 변수 설정
            linear2 = self.linear_2(relu) # 변수 설정
            output = self.sigmoid(linear2) # 변수 설정
            return output # 결과 반환

In [ ]:
model = NeuralNet(2, 5) # 변수 설정
learning_rate = 0.03 # 학습률 설정
criterion = torch.nn.BCELoss() # 손실 함수 정의
epochs = 2000 # 에포크 설정
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate) # 최적화 알고리즘 설정

In [ ]:
model.eval() # 코드 실행
test_loss_before = criterion(model(x_test).squeeze(), y_test) # 모델 예측
print('Before Training, test loss is {}'.format(test_loss_before.item())) # 결과 출력

In [ ]:
for epoch in range(epochs): # 반복문 수행
    model.train() # 코드 실행
    optimizer.zero_grad() # 기울기 초기화
    train_output = model(x_train) # 모델 예측
    train_loss = criterion(train_output.squeeze(), y_train) # 변수 설정
    if epoch % 100 == 0: # 조건문 수행
        print('Train loss at {} is {}'.format(epoch, train_loss.item())) # 결과 출력
    train_loss.backward() # 역전파 수행
    optimizer.step() # 가중치 업데이트

In [ ]:
model.eval() # 코드 실행
test_loss = criterion(torch.squeeze(model(x_test)), y_test) # 모델 예측
print('After Training, test loss is {}'.format(test_loss.item())) # 결과 출력

In [ ]:
torch.save(model.state_dict(), './model.pt') # 모델 저장
print('state_dict format of the model: {}'.format(model.state_dict())) # 결과 출력

In [ ]:
new_model = NeuralNet(2, 5) # 변수 설정
new_model.load_state_dict(torch.load('./model.pt')) # 모델 파라미터 불러오기
new_model.eval() # 코드 실행
print('벡터 [-1, 1]이 레이블 1을 가질 확률은 {}'.format(new_model(torch.FloatTensor([-1,1])).item())) # 텐서 변환